In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
import matplotlib.pyplot as plt

# Anchor to the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from src.config import SimConfig, EnvConfig
from src.utils.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker, print_markdown_table
from src.solvers import AugmentedHybridSDPSolver
from src.plants import AugmentedHybridPlant
from src.controllers import build_approach, AugmentedValueControl, AugmentedPolicyControl, AugmentedFCLockedControl

In [ ]:
# 1. Setup the Environment and Load Data
env = EnvConfig()
fleet_data = load_and_cache_entire_fleet(env)

# Using the full 11 days for a rigorous scientific average
exclude_days = [1,2,3] 

# [CRITICAL STEP] 
# Insert the optimal 'elbow' dP you discovered in Notebook 2 here!
# For this script, we will assume 140 kW was the sweet spot.
OPTIMAL_DP = 150.0

In [ ]:
# 2. Define the Test Vector for Temporal Resolution (Macro-step length in seconds)
# Ranging from a highly reactive 1-minute step to a sluggish 15-minute step
dt_test_values = range(60, 601, 60)

results_data = []

print("--- RUNNING TEMPORAL RESOLUTION SENSITIVITY ---")

for dt_val in dt_test_values:
    print(f"\n[ Evaluating Grid: Dt = {dt_val} seconds ]")
    
    # Instantiate config locking dP, varying Dt
    config = SimConfig(
        dP=OPTIMAL_DP, 
        Dt=dt_val, 
        N_Pd=6, 
        use_smart_grid=True, 
        alpha_fc=4,
        n_pack=4
    )
    
    benchmarker = VoyageBenchmarker(fleet_data, env, config, exclude_days)
    
    # Strictly use the winning continuous Value Control architecture
    value_real_factory = build_approach(
        controller_cls=AugmentedFCLockedControl, 
        plant_cls=AugmentedHybridPlant, 
        solver_cls=AugmentedHybridSDPSolver, 
        is_macro=False
    )
    
    # Run the benchmark
    report = benchmarker.run_leave_one_out(value_real_factory)
    avg_metrics = report.summary.loc['Average']
    
    # Log the metrics for the Pareto front
    results_data.append({
        'Dt [s]': dt_val,
        'Average Total Cost [$]': avg_metrics['Total Cost [$]'],
        'Offline Compute Time [s]': avg_metrics['Offline Compute Time [s]'],
        'Online Compute Time [s]': avg_metrics['Online Compute Time [s]']
    })

# Convert to DataFrame
df_results = pd.DataFrame(results_data).set_index('Dt [s]')

In [ ]:
print("\n--- TEMPORAL RESOLUTION: SENSITIVITY SUMMARY ---")
print_markdown_table(df_results)

In [ ]:
# 3. Plot the L-Shaped Pareto Curve for the Article
fig, ax = plt.subplots(figsize=(9, 6))

# Extract data
compute_times = df_results['Offline Compute Time [s]'].values
total_costs = df_results['Average Total Cost [$]'].values
dt_labels = df_results.index.values

# Plot the curve (We reverse the order visually so the graph flows left to right as Dt gets smaller/faster)
ax.plot(compute_times, total_costs, marker='s', linestyle='-', color='crimson', linewidth=2.5, markersize=8)

# Annotate each point with its Dt value
for i, dt_val in enumerate(dt_labels):
    ax.annotate(f"Dt={int(dt_val)}s", 
                (compute_times[i], total_costs[i]), 
                xytext=(10, 5), 
                textcoords='offset points',
                fontsize=10, 
                fontweight='bold', 
                color='darkred')

# Formatting for academic standards
ax.set_title("Pareto Frontier: Temporal Resolution vs. Offline Complexity", fontsize=14, fontweight='bold')
ax.set_xlabel("Offline Compute Time [Seconds]", fontsize=12)
ax.set_ylabel("Average Total Cost [$]", fontsize=12)
ax.grid(True, linestyle='--', alpha=0.6)

# Save and Show
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/pareto_temporal_resolution.png', dpi=300, bbox_inches='tight')
plt.show()

# Print the complexity proof for Dt scaling
print("\nComplexity Check:")
print(f"Ratio of Compute Time (Dt=150s vs Dt=300s): {compute_times[1] / compute_times[2]:.2f}x")
print(f"Theoretical O(1/Dt^2) Ratio: {(300**2) / (150**2):.2f}x")